In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 데이터 전처리

In [ ]:
import os
import pandas as pd
import pickle
import openpyxl

file_path = 'drive/MyDrive/데이터사이언스/goal_set.p'
goal_set = pickle.load(open(file_path, 'rb'))

train_data = goal_set["train"]
test_data = goal_set["test"]

df_train = pd.DataFrame(train_data)
df_test = pd.DataFrame(test_data)

df_train["explicit_inform_slots"] = df_train["goal"].apply(lambda x: f"'{next(iter(x.get('explicit_inform_slots', {}).keys()), '')}'")
df_train["implicit_inform_slots"] = df_train["goal"].apply(lambda x: [f"'{key}'" for key in x.get('implicit_inform_slots', {}).keys()])
df_train['combined_slots'] = df_train.apply(lambda row: [row['explicit_inform_slots']] + row['implicit_inform_slots'], axis=1)

df_test["explicit_inform_slots"] = df_test["goal"].apply(lambda x: f"'{next(iter(x.get('explicit_inform_slots', {}).keys()), '')}'")
df_test["implicit_inform_slots"] = df_test["goal"].apply(lambda x: [f"'{key}'" for key in x.get('implicit_inform_slots', {}).keys()])
df_test['combined_slots'] = df_test.apply(lambda row: [row['explicit_inform_slots']] + row['implicit_inform_slots'], axis=1)

# 작은따옴표를 제거하는 함수 정의
def remove_quotes(lst):
    return [item.replace("'", "") for item in lst]

df_train['combined_slots'] = df_train['combined_slots'].apply(remove_quotes)
df_test['combined_slots'] = df_test['combined_slots'].apply(remove_quotes)

# 필요 없는 열 제거
df_train.drop(columns=['explicit_inform_slots', 'implicit_inform_slots', 'goal', 'group_id'], inplace=True)
df_test.drop(columns=['explicit_inform_slots', 'implicit_inform_slots', 'goal', 'group_id'], inplace=True)

df_test.drop(columns='consult_id', inplace=True)
df_train.drop(columns='consult_id', inplace=True)

# combined_slots에서 모든 값을 모아서 유일한 값들을 출력합니다.
unique_train = set()
for slots_list in df_train['combined_slots']:
    unique_train.update(slots_list)

unique_train = list(unique_train)

# combined_slots에서 모든 값을 모아서 유일한 값들을 출력합니다.
unique_test = set()
for slots_list in df_test['combined_slots']:
    unique_test.update(slots_list)

unique_test = list(unique_test)

# 중복되지 않는 값을 찾습니다.
unique_train_not_in_test = [item for item in unique_train if item not in unique_test]

# 대상 문자열
target_strings_train = [
    'Skin dryness, peeling, scaliness, or roughness',
    'Muscle cramps, contractures, or spasms'
]

# 'combined_slots' 열의 각 리스트 값에 대해 반복하면서 쉼표 없애기
for i, slots_list in enumerate(df_train['combined_slots']):
    updated_slots_list = []
    for item in slots_list:
        # 대상 문자열이 있는 경우 쉼표 없애고 업데이트
        if item in target_strings_train:
            updated_slots_list.append(item.replace(',', ''))
        else:
            updated_slots_list.append(item)
    # 업데이트된 리스트로 교체
    df_train.at[i, 'combined_slots'] = updated_slots_list

target_strings_test = [
    'Skin dryness, peeling, scaliness, or roughness'
]

# 'combined_slots' 열의 각 리스트 값에 대해 반복하면서 쉼표 없애기
for i, slots_list in enumerate(df_test['combined_slots']):
    updated_slots_list = []
    for item in slots_list:
        # 대상 문자열이 있는 경우 쉼표 없애고 업데이트
        if item in target_strings_train:
            updated_slots_list.append(item.replace(',', ''))
        else:
            updated_slots_list.append(item)
    # 업데이트된 리스트로 교체
    df_test.at[i, 'combined_slots'] = updated_slots_list

train = df_train.copy()
test  = df_test.copy()

train.to_excel('train.xlsx', index=False)
test.to_excel('test.xlsx', index=False)

# TF-IDF 벡터화

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# 쉼표를 기준으로 문서를 토큰화하는 사용자 정의 토크나이저 함수
def custom_tokenizer(text):
    # 쉼표를 기준으로 분할한 뒤 앞뒤 공백을 제거하여 반환
    return [word.strip() for word in text.split(', ')]

# TF-IDF 벡터화를 위한 객체 생성 (쉼표를 기준으로 토큰화하는 사용자 정의 토크나이저 지정)
tfidf_vectorizer = TfidfVectorizer(tokenizer=custom_tokenizer)

# 'combined_slots' 열의 값을 하나의 텍스트 문서로 합치기
documents_train = df_train['combined_slots'].apply(lambda x: ', '.join(x))

# TF-IDF 벡터화를 수행하고 결과를 반환
tfidf_matrix_train = tfidf_vectorizer.fit_transform(documents_train)

# TF-IDF 벡터화된 결과를 데이터프레임으로 변환하여 출력
df_tfidf_train = pd.DataFrame(tfidf_matrix_train.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
df_tfidf_train.set_index(df_train['disease_tag'], inplace=True)

# 'combined_slots' 열의 값을 하나의 텍스트 문서로 합치기
documents_test = df_test['combined_slots'].apply(lambda x: ', '.join(x))

# TF-IDF 벡터화를 수행하고 결과를 반환
tfidf_matrix_test = tfidf_vectorizer.transform(documents_test)

# TF-IDF 벡터화된 결과를 데이터프레임으로 변환하여 출력
df_tfidf_test = pd.DataFrame(tfidf_matrix_test.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
df_tfidf_test.set_index(df_test['disease_tag'], inplace=True)

/usr/local/lib/python3.10/dist-packages/sklearn/feature_extraction/text.py:528: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


# 질병 예측 모델

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 학습 데이터와 타겟 데이터 분리
X_train = df_tfidf_train
y_train = df_tfidf_train.index

X_test = df_tfidf_test
y_test = df_tfidf_test.index

# 로지스틱 회귀 모델 학습
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 검증 데이터에 대한 예측
y_pred = model.predict(X_test)

# 모델 평가
print(classification_report(y_test, y_pred))

                                          precision    recall  f1-score   support

                    Acanthosis nigricans       0.77      0.90      0.83        61
                               Acariasis       0.73      0.92      0.81        72
                                    Acne       0.84      0.96      0.90        73
                       Actinic keratosis       0.69      0.80      0.74        79
                          Acute glaucoma       0.30      0.36      0.33        58
                     Acute kidney injury       0.86      0.80      0.83        75
                   Acute stress reaction       0.92      0.69      0.79        70
     Adhesive capsulitis of the shoulder       0.87      0.98      0.92        54
                     Adjustment reaction       0.71      0.65      0.68        83
                            Air embolism       0.45      0.07      0.13        67
                    Alcohol intoxication       0.88      0.88      0.88        68
               

In [ ]:
# 입력 증상 TF-IDF 벡터화 함수
def transform_input_symptoms(input_symptoms, tfidf_vectorizer):
    input_text = ' '.join(input_symptoms)
    input_vector = tfidf_vectorizer.transform([input_text])
    return input_vector

# 예측 함수
def predict_top_diseases(input_symptoms, tfidf_vectorizer, model, top_n=5):
    input_vector = transform_input_symptoms(input_symptoms, tfidf_vectorizer)
    probabilities = model.predict_proba(input_vector)[0]
    top_indices = probabilities.argsort()[-top_n:][::-1]
    top_diseases = [model.classes_[i] for i in top_indices]
    top_probabilities = [probabilities[i] for i in top_indices]
    return list(zip(top_diseases, top_probabilities))

# 입력된 증상으로부터 상위 N개의 질병 예측
input_symptoms = ['abnormal appearing skin', 'wrist lump or mass']
top_diseases = predict_top_diseases(input_symptoms, tfidf_vectorizer, model, top_n=5)
print(f"Top {len(top_diseases)} predicted diseases for input '{input_symptoms}':")
for disease, probability in top_diseases:
    print(f"{disease}: {probability:.4f}")

Top 5 predicted diseases for input '['abnormal appearing skin', 'wrist lump or mass']':
Chancroid: 0.0419
Chagas disease: 0.0398
Graves disease: 0.0271
Diabetic retinopathy: 0.0261
Cystic Fibrosis: 0.0247


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


#TF-IDF 클러스터링 방법 (증상 - 증상)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# 쉼표를 기준으로 문서를 토큰화하는 사용자 정의 토크나이저 함수
def custom_tokenizer(text):
    # 쉼표를 기준으로 분할한 뒤 앞뒤 공백을 제거하여 반환
    return [word.strip() for word in text.split(', ')]

# TF-IDF 벡터화를 위한 객체 생성 (쉼표를 기준으로 토큰화하는 사용자 정의 토크나이저 지정)
tfidf_vectorizer = TfidfVectorizer(tokenizer=custom_tokenizer)

# 'combined_slots' 열의 값을 하나의 텍스트 문서로 합치기
documents_train = df_train['combined_slots'].apply(lambda x: ', '.join(x))

# TF-IDF 벡터화를 수행하고 결과를 반환
tfidf_matrix_train = tfidf_vectorizer.fit_transform(documents_train)

# TF-IDF 벡터화된 결과를 데이터프레임으로 변환하여 출력
df_tfidf_train = pd.DataFrame(tfidf_matrix_train.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
df_tfidf_train.set_index(df_train['disease_tag'], inplace=True)

# 'combined_slots' 열의 값을 하나의 텍스트 문서로 합치기
documents_test = df_test['combined_slots'].apply(lambda x: ', '.join(x))

# TF-IDF 벡터화를 수행하고 결과를 반환
tfidf_matrix_test = tfidf_vectorizer.transform(documents_test)

# TF-IDF 벡터화된 결과를 데이터프레임으로 변환하여 출력
df_tfidf_test = pd.DataFrame(tfidf_matrix_test.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
df_tfidf_test.set_index(df_test['disease_tag'], inplace=True)

/usr/local/lib/python3.10/dist-packages/sklearn/feature_extraction/text.py:528: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [ ]:
df_train_group = df_tfidf_train.copy()
df_train_group = df_train_group.groupby('disease_tag').mean()
df_test_group = df_tfidf_test.copy()
df_test_group = df_test_group.groupby('disease_tag').mean()

In [ ]:
df_train_group_T = df_train_group.transpose()
df_test_group_T = df_test_group.transpose()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

vectors_train = df_train_group_T.values
vectors_test = df_test_group_T.values

# 코사인 유사도를 계산합니다.
cosine_similarities_train = cosine_similarity(vectors_train)
cosine_similarities_test = cosine_similarity(vectors_test)

# 결과를 데이터프레임으로 변환합니다.
cosine_similarities_train = pd.DataFrame(cosine_similarities_train, columns=df_train_group_T.index, index=df_train_group_T.index)
cosine_similarities_test = pd.DataFrame(cosine_similarities_test, columns=df_test_group_T.index, index=df_test_group_T.index)

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
import numpy as np
from scipy.spatial.distance import cdist

# K-Means 클러스터링 수행
k = 6 # 클러스터 개수, 필요에 따라 조정
kmeans = KMeans(n_clusters=k, random_state=42).fit(cosine_similarities_train)

# 클러스터 레이블을 데이터프레임에 추가
cosine_similarities_train['cluster'] = kmeans.labels_

# 클러스터 중심 계산
cluster_centers = kmeans.cluster_centers_

# 함수: 입력 증상 기반으로 관련 증상 찾기
def find_related_symptoms(input_symptoms, df_cosine_sim, num_related=10):
    related_symptoms = []
    clusters = df_cosine_sim.loc[input_symptoms, 'cluster'].unique()

    for cluster in clusters:
        cluster_symptoms = df_cosine_sim[df_cosine_sim['cluster'] == cluster].index
        related_symptoms.extend(cluster_symptoms)

    # 입력 증상 제외
    related_symptoms = list(set(related_symptoms) - set(input_symptoms))

    # 관련 증상을 코사인 유사도 순으로 정렬하여 상위 num_related개 반환
    related_symptoms_scores = [(symptom, df_cosine_sim.loc[input_symptoms, symptom].mean()) for symptom in related_symptoms]
    related_symptoms_scores = sorted(related_symptoms_scores, key=lambda x: x[1], reverse=True)

    # num_related 개수만큼 반환, 부족할 경우 가까운 클러스터에서 채움
    result = [symptom for symptom, score in related_symptoms_scores[:num_related]]
    if len(result) < num_related:
        # 입력 증상 클러스터의 중심 좌표
        input_cluster_centroid = np.mean(cluster_centers[clusters], axis=0)

        # 모든 클러스터 중심과의 거리 계산
        distances = cdist([input_cluster_centroid], cluster_centers, 'euclidean')[0]

        # 가까운 클러스터 순서로 정렬
        nearest_clusters = np.argsort(distances)

        for cluster in nearest_clusters:
            if len(result) >= num_related:
                break
            if cluster in clusters:
                continue  # 이미 처리한 클러스터는 건너뜀

            cluster_symptoms = df_cosine_sim[df_cosine_sim['cluster'] == cluster].index
            additional_symptoms = list(set(cluster_symptoms) - set(input_symptoms) - set(result))

            additional_symptoms_scores = [(symptom, df_cosine_sim.loc[input_symptoms, symptom].mean()) for symptom in additional_symptoms]
            additional_symptoms_scores = sorted(additional_symptoms_scores, key=lambda x: x[1], reverse=True)
            result.extend([symptom for symptom, score in additional_symptoms_scores])

    return result[:num_related]

/usr/local/lib/python3.10/dist-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


In [ ]:
# 예시: 입력 증상 리스트
input_symptoms = ['vomiting']



# 관련 증상 찾기
related_symptoms = find_related_symptoms(input_symptoms, cosine_similarities_train)
print("입력 증상:", input_symptoms)
print("관련 증상:", related_symptoms)

입력 증상: ['vomiting']
관련 증상: ['nausea']


In [ ]:
import math
import numpy as np

def dcg(rel,i) :
    return rel/(math.log2(i+1))

def idcg(rel,t) :
    if t==0 :
        return 0
    else :
        return dcg(rel,t)+idcg(rel,t-1)

def ndcg_n_cal (p,label,n) :
    if n==0 :
        return 0
    elif n<=len(p) and p[n-1] in label :
        return dcg(1,n)+ndcg_n_cal(p,label,n-1)
    else:
        return ndcg_n_cal(p,label,n-1)

def ndcg_n(p,label,n) :
    return ndcg_n_cal(p,label,n) / idcg(1,len(label))

def do_ndcg(input,label,n) :
  if('skin dryness, peeling, scaliness, or roughness' in label or 'skin dryness, peeling, scaliness, or roughness' in input
     or ['muscle cramps, contractures, or spasms'] in label or ['muscle cramps, contractures, or spasms'] in input or
     'muscle cramps, contractures, or spasms' in label or 'muscle cramps, contractures, or spasms' in input) :
    return 0
  top_diseases = find_related_symptoms(input, cosine_similarities_train)
  print(top_diseases)
  ndcg_score=ndcg_n(top_diseases,label,n)
  return ndcg_score

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/Task1.csv')
results = []
n=10
for index, row in data.iterrows():
    input_disease = eval(row['explicit_inform_slots'])
    answer_disease = eval(row['implicit_inform_slots'])  # Convert the string representation of list to actual list
    print(input_disease)
    print(answer_disease)
    ndcg_score = do_ndcg(input_disease, answer_disease, 10)
    print(ndcg_score)
    results.append(ndcg_score)

print(results)
print("Task1의 NDCG@10 결과 : ",np.mean(results))

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
['groin pain', 'neck cramps or spasms', 'shoulder swelling', 'vulvar sore', 'excessive growth', 'nailbiting', 'wrist weakness', 'sharp chest pain', 'loss of sex drive', 'nausea']
0.0
['shoulder pain']
['elbow cramps or spasms', 'itching of scrotum']
['shoulder stiffness or tightness', 'ankle stiffness or tightness', 'skin on arm or hand looks infected', 'stiffness all over', 'ache all over', 'arm pain', 'bladder mass', 'decreased heart rate', 'itching of scrotum', 'muscle swelling']
0.18457569677956817
['low back pain']
['ache all over', 'arm pain']
['back pain', 'lower body pain', 'leg pain', 'neck pain', 'hip pain', 'ache all over', 'back cramps or spasms', 'back stiffness or tightness', 'stiffness all over', 'elbow pain']
0.21840743681816419
['heavy menstrual flow']
['vaginal bleeding after menopause']
['abnormal size or shape of ear', 'pus in sputum', 'unpredictable menstruation', 'vaginal bleeding after menopause', 'pelvic pain', 'low back weakn